# Olist Market Research Dashboard Data Preparation
---
## Project Background
### About Olist
Olist is a Brazil-based company that offers solutions for online sales, marketplace integration, and business management tools. Olist connects sellers to multiple e-commerce platforms, helping them track SKU movements, manage logistics, and handle digital payments across online platforms.

### Dataset Information
This dataset is provided by Olist and uploaded to Kaggle as a public dataset. The dataset has information on 100k orders from 2016 to 2018 made at multiple marketplaces in Brazil. Its features allow viewing an order from multiple dimensions: from order status, price, payment, and freight performance to customer location, product attributes, and finally reviews written by customers.

### Project Scope
For this part of the process, we will work mainly on the `olist_customers_dataset.csv`. This CSV file holds the customers' geospatial details that are required for the market entry analysis. However, the current `customer_state` column only holds the abbreviated version of each state. This is a mismatch for the native `Filled Map` visual in Power BI, which requires the full name of each state. 

This notebook will document the data cleaning/feature engineering process to match Power BI's native `Filled Map` value requirements.

In [1]:
# Importing key packages
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_customers_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_products_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/product_category_name_translation.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_orders_dataset.csv
/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv


In [2]:
# Reading the CSV into a pandas DataFrame
df = pd.read_csv('/kaggle/input/datasets/olistbr/brazilian-ecommerce/olist_customers_dataset.csv')

# Checking the DataFrame
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [3]:
# Checking the first 5 rows of the DataFrame
df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [4]:
# Checking the number of states and the name of each states
# There are 26 states and 1 federal district (there should be 27 unique values present)
print(len(df['customer_state'].unique()))
print(df['customer_state'].unique())

27
['SP' 'SC' 'MG' 'PR' 'RJ' 'RS' 'PA' 'GO' 'ES' 'BA' 'MA' 'MS' 'CE' 'DF'
 'RN' 'PE' 'MT' 'AM' 'AP' 'AL' 'RO' 'PB' 'TO' 'PI' 'AC' 'SE' 'RR']


In [5]:
# This CSV is copied from Brave Search's AI Answer
# The values in this CSV have accented/diacritic characters, which will not match Power BI's expected values

abbr = '''"State","Abbreviation","Capital"
"Acre","AC","Rio Branco"
"Alagoas","AL","Maceió"
"Amapá","AP","Macapá"
"Amazonas","AM","Manaus"
"Bahia","BA","Salvador"
"Ceará","CE","Fortaleza"
"Distrito Federal","DF","Brasília"
"Espírito Santo","ES","Vitória"
"Goiás","GO","Goiânia"
"Maranhão","MA","São Luís"
"Mato Grosso","MT","Cuiabá"
"Mato Grosso do Sul","MS","Campo Grande"
"Minas Gerais","MG","Belo Horizonte"
"Pará","PA","Belém"
"Paraíba","PB","João Pessoa"
"Paraná","PR","Curitiba"
"Pernambuco","PE","Recife"
"Piauí","PI","Teresina"
"Rio de Janeiro","RJ","Rio de Janeiro"
"Rio Grande do Norte","RN","Natal"
"Rio Grande do Sul","RS","Porto Alegre"
"Rondônia","RO","Porto Velho"
"Roraima","RR","Boa Vista"
"Santa Catarina","SC","Florianópolis"
"São Paulo","SP","São Paulo"
"Sergipe","SE","Aracaju"
"Tocantins","TO","Palmas"'''

In [6]:
# Using the StringIO and unicodedata modules to read the string as a CSV, and normalize the accented/diacritical characters

from io import StringIO
import unicodedata

clean = unicodedata.normalize('NFD', abbr).encode('ascii', 'ignore').decode('utf-8')
states = pd.read_csv(StringIO(clean))
states.head()

,State,Abbreviation,Capital
0,Acre,AC,Rio Branco
1,Alagoas,AL,Maceio
2,Amapa,AP,Macapa
3,Amazonas,AM,Manaus
4,Bahia,BA,Salvador


In [7]:
# Mapping the cleaned state name via dict values, referencing the abbreviated state names in `customer_state`

dict_abb = states.set_index('Abbreviation')['State'].to_dict()

df['state_name'] = df['customer_state'].map(dict_abb)

In [8]:
# Checking new dataframe structure (should have a new column) 
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
 5   state_name                99441 non-null  object
dtypes: int64(1), object(5)
memory usage: 4.6+ MB


In [9]:
# Checking the first 5 rows of the DataFrame (should have a normalized state name)
df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,state_name
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,Sao Paulo
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,Sao Paulo
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,Sao Paulo
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,Sao Paulo
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,Sao Paulo


In [10]:
# Checking the number of states and the name of each states
# There are 26 states and 1 federal district (there should be 27 unique values present)
print(len(df['state_name'].unique()))
print(df['state_name'].unique())

27
['Sao Paulo' 'Santa Catarina' 'Minas Gerais' 'Parana' 'Rio de Janeiro'
 'Rio Grande do Sul' 'Para' 'Goias' 'Espirito Santo' 'Bahia' 'Maranhao'
 'Mato Grosso do Sul' 'Ceara' 'Distrito Federal' 'Rio Grande do Norte'
 'Pernambuco' 'Mato Grosso' 'Amazonas' 'Amapa' 'Alagoas' 'Rondonia'
 'Paraiba' 'Tocantins' 'Piaui' 'Acre' 'Sergipe' 'Roraima']


In [11]:
# Exporting the new dataset to the working directory for download
df.to_csv('/kaggle/working/olist_customers_dataset_new.csv')